In [2]:
import json
import urllib.request
from collections import Counter, defaultdict
from textwrap import shorten

import numpy as np
import pandas as pd

fp = "https://openaipublic.blob.core.windows.net/simple-evals/healthbench/2025-05-07-06-14-12_oss_eval.jsonl"
records = []

with urllib.request.urlopen(fp) as f:
    for line in f:
        records.append(json.loads(line))

example_rows = []
rubric_rows = []
for example_index, ex in enumerate(records):
    example_tags = ex.get("example_tags", [])
    themes = [tag.removeprefix("theme:") for tag in example_tags if tag.startswith("theme:")]
    categories = [
        tag.removeprefix("physician_agreed_category:")
        for tag in example_tags
        if tag.startswith("physician_agreed_category:")
    ]
    prompt = ex.get("prompt", [])
    prompt_text = " ".join(turn.get("content", "") for turn in prompt)
    example_rows.append({
        "example_index": example_index,
        "prompt_id": ex.get("prompt_id"),
        "themes": themes,
        "categories": categories,
        "n_turns": len(prompt),
        "prompt_chars": len(prompt_text),
        "prompt_text": prompt_text,
        "n_rubrics": len(ex.get("rubrics", [])),
    })

    for rubric_index, rubric in enumerate(ex.get("rubrics", [])):
        tags = rubric.get("tags", [])
        rubric_rows.append({
            "example_index": example_index,
            "rubric_index": rubric_index,
            "criterion": rubric.get("criterion", ""),
            "points": rubric.get("points"),
            "axes": [tag.removeprefix("axis:") for tag in tags if isinstance(tag, str) and tag.startswith("axis:")],
            "levels": [tag.removeprefix("level:") for tag in tags if isinstance(tag, str) and tag.startswith("level:")],
        })

examples_df = pd.DataFrame(example_rows)
rubrics_df = pd.DataFrame(rubric_rows)

print(f"Examples: {len(examples_df):,}")
print(f"Rubric attachments: {len(rubrics_df):,}")
print(f"Unique rubric criteria: {rubrics_df['criterion'].nunique():,}")
print("\nTheme counts:")
print(examples_df.explode("themes")["themes"].value_counts().to_string())
print("\nAxis counts:")
print(rubrics_df.explode("axes")["axes"].value_counts().to_string())
print("\nLevel counts:")
print(rubrics_df.explode("levels")["levels"].value_counts().to_string())
print("\nShape quantiles:")
print(examples_df[["n_turns", "prompt_chars", "n_rubrics"]].quantile([0, .25, .5, .75, .95, 1]))

Examples: 5,000
Rubric attachments: 57,237
Unique rubric criteria: 48,562

Theme counts:
themes
global_health          1097
hedging                1071
communication           919
context_seeking         594
emergency_referrals     482
health_data_tasks       477
complex_responses       360

Axis counts:
axes
completeness             22285
accuracy                 18888
context_awareness         8991
communication_quality     4522
instruction_following     2551

Level counts:
levels
example    49184
cluster     8053

Shape quantiles:
      n_turns  prompt_chars  n_rubrics
0.00      1.0          4.00        2.0
0.25      1.0         98.00        8.0
0.50      1.0        281.00       11.0
0.75      3.0        953.25       15.0
0.95      7.0       2428.20       21.0
1.00     19.0       9859.00       48.0


# HealthBench label guide

This notebook has **three different labeling layers**:

- **Theme** labels describe the main capability or challenge of the *whole patient prompt*. Each example has a `theme:` tag.
- **Axis** labels describe what a *rubric criterion is evaluating*: whether the answer is correct, complete, context-sensitive, well communicated, or follows the requested instructions. A rubric can carry more than one axis.
- **Level** labels describe the *scope of a rubric criterion*, not clinical severity. `level:example` is specific to the individual prompt; `level:cluster` is intended to capture a reusable criterion shared across a related group of prompts.

The descriptions below are **empirical interpretations of the labels in this release**, synthesized from their criteria and prompts. They should be read as operational meanings, not as a claim that the dataset publishes a formal dictionary.

## Theme labels: what kind of challenge is this?

| Theme | Operational meaning |
|---|---|
| `global_health` | The answer must reason across geography, health systems, resource constraints, epidemiology, culture, or population-level access. |
| `hedging` | The task tests calibrated uncertainty: avoid overclaiming, distinguish possibilities from diagnoses, state limitations, and explain when evaluation or escalation is needed. |
| `communication` | The task tests whether advice is understandable, empathetic, respectful, and tailored to the patient's likely needs rather than merely technically correct. |
| `context_seeking` | Important information is missing or ambiguous, so a strong answer asks targeted follow-up questions before giving overly specific advice. |
| `emergency_referrals` | The scenario contains danger signs or an acute risk where prompt emergency care, escalation, or a safety plan must be prioritized. |
| `health_data_tasks` | The user asks for a data-oriented health task such as interpreting measurements, organizing information, calculating a risk, or working with a record or result. |
| `complex_responses` | The response must integrate multiple constraints, steps, conditions, or clinical considerations instead of answering with one isolated fact. |

## Axis labels: what quality dimension is being scored?

| Axis | Operational meaning |
|---|---|
| `accuracy` | Factual and clinical correctness: recommendations, calculations, interpretations, contraindications, and distinctions should be right. |
| `completeness` | Cover the important parts of the task, including required steps, warnings, alternatives, follow-up, and contingencies. |
| `context_awareness` | Use the details supplied by the patient and recognize what missing context changes the answer; avoid generic advice that ignores the case. |
| `communication_quality` | Present the answer clearly and appropriately, with useful organization, plain language, empathy, and proportional emphasis. |
| `instruction_following` | Satisfy the explicit format, scope, role, or output requirements in the user's request. |

## Level buckets: how reusable is the criterion?

| Level | Operational meaning |
|---|---|
| `example` | A criterion tied to this particular prompt and its expected answer. It often mentions the case's specific symptom, measurement, decision, or safety action. |
| `cluster` | A criterion designed to recur across a group of related prompts. It tends to express a broader pattern such as being transparent about uncertainty, checking contraindications, or giving an appropriate escalation plan. |

The next cells test these interpretations against the actual criterion text, show axis-by-level counts, and print representative examples for every label.

In [3]:
# Cross-tabs and representative criteria

axis_level = pd.crosstab(
    rubrics_df.explode("axes")["axes"],
    rubrics_df.explode("levels")["levels"],
)
print("Axis x level counts:")
print(axis_level.to_string())

print("\nTheme x rubric-axis prevalence (% of examples with at least one criterion on the axis):")
theme_axis_rows = []
for theme in sorted(examples_df.explode("themes")["themes"].dropna().unique()):
    theme_examples = examples_df[examples_df["themes"].apply(lambda labels: theme in labels)]
    indices = set(theme_examples["example_index"])
    theme_rubrics = rubrics_df[rubrics_df["example_index"].isin(indices)]
    for axis in sorted({axis for labels in theme_rubrics["axes"] for axis in labels}):
        axis_indices = set(
            theme_rubrics[theme_rubrics["axes"].apply(lambda labels: axis in labels)]["example_index"]
        )
        theme_axis_rows.append({
            "theme": theme,
            "axis": axis,
            "example_prevalence_pct": round(100 * len(axis_indices) / len(theme_examples), 1),
        })
print(pd.DataFrame(theme_axis_rows).pivot(index="theme", columns="axis", values="example_prevalence_pct").fillna(0).round(1).to_string())


def representative_criteria(label_column, labels):
    print(f"\nRepresentative {label_column} criteria:")
    for label in labels:
        if label_column == "themes":
            matching_examples = examples_df[examples_df["themes"].apply(lambda values: label in values)]
            matching = rubrics_df[rubrics_df["example_index"].isin(matching_examples["example_index"])].copy()
        else:
            matching = rubrics_df[rubrics_df[label_column].apply(lambda values: label in values)].copy()
        matching["criterion_length"] = matching["criterion"].str.len()
        matching = matching.sort_values("criterion_length")
        row = matching.iloc[len(matching) // 2]
        example = examples_df.loc[examples_df["example_index"] == row["example_index"]].iloc[0]
        print(f"\n{label} (n={len(matching):,})")
        print("  Prompt:", shorten(example["prompt_text"], width=260, placeholder=" ..."))
        print("  Criterion:", row["criterion"])
        print("  Points:", row["points"], "| axes:", row["axes"], "| levels:", row["levels"])

print(representative_criteria("themes", sorted({theme for labels in examples_df["themes"] for theme in labels}))
)
print(representative_criteria("axes", sorted({axis for labels in rubrics_df["axes"] for axis in labels})))
print(representative_criteria("levels", sorted({level for labels in rubrics_df["levels"] for level in labels})))

Axis x level counts:
levels                 cluster  example
axes                                   
accuracy                  3469    15419
communication_quality     1704     2818
completeness               319    21966
context_awareness         1986     7005
instruction_following      575     1976

Theme x rubric-axis prevalence (% of examples with at least one criterion on the axis):
axis                 accuracy  communication_quality  completeness  context_awareness  instruction_following
theme                                                                                                       
communication            97.1                   87.4          79.2               40.9                   22.2
complex_responses        97.5                   94.2          83.1               44.7                   53.6
context_seeking          92.4                   31.6          95.5               85.0                   21.0
emergency_referrals      66.0                   32.4          89.

## What the EDA suggests

- **Themes are prompt-level challenge families, not mutually exclusive clinical topics.** The most common are `global_health` (1,097 examples), `hedging` (1,071), and `communication` (919). The theme counts sum to 5,000 because the release assigns one theme per example.
- **Axes are rubric-level dimensions.** `completeness` is the most frequent axis (22,285 rubric attachments), followed by `accuracy` (18,888). These counts are larger than the number of examples because one prompt can have many criteria.
- **The most diagnostic theme-axis relationships are sensible.** `emergency_referrals` has context-awareness criteria in 96.1% of its examples; `health_data_tasks` has instruction-following criteria in 91.8%; and `context_seeking` has context-awareness criteria in 85.0%.
- **Levels encode criterion scope.** `example` criteria dominate (49,184 attachments) and usually refer to a particular symptom, result, or requested action. `cluster` criteria are fewer (8,053) and often read like reusable policies, such as requiring factual correctness and sufficient safety coverage across a family of tasks.
- **The benchmark does not expose a one-sentence official definition for every theme.** The label meanings above are therefore reverse-engineered from the label names, their distributions, and their associated prompts and criteria. The representative outputs are intended to make those operational meanings inspectable.